# 🧠 EX55: โหมดการประเมินผล (Evaluation Modes & Detection Metrics)

## IoU — Intersection over Union
$$\text{IoU}(\hat{B}, B_{gt}) = \frac{|\hat{B}\cap B_{gt}|}{|\hat{B}\cup B_{gt}|}$$
- IoU ≥ 0.5 → TP (ตรวจจับถูก)  |  IoU < 0.5 → FP (ตรวจจับผิด)

## Precision & Recall
$$P = \frac{TP}{TP+FP} \qquad R = \frac{TP}{TP+FN}$$

## Average Precision (AP)
101-point COCO interpolation ของ PR curve:
$$\text{AP} = \sum_k (R_k - R_{k-1})\,P_k$$

## mAP@0.5 vs mAP@0.5:0.95
| ตัวชี้วัด | ความหมาย |
|----------|---------|
| mAP@0.5 | ค่าเฉลี่ย AP ทุกคลาส @ IoU=0.50 |
| mAP@0.5:0.95 | ค่าเฉลี่ย AP ที่ IoU 0.50–0.95 (มาตรฐาน COCO) |

mAP@0.5 สูง + mAP@0.5:0.95 ต่ำ = กล่องขอบเขตไม่แม่นยำ

## 🔗 ลิงก์
- [[EX54_Resuming_Training_TH]] | [[YOLO_Learning_Plan]]


In [ ]:
# Back up or checkpoint this section of code before starting to modify the large file.
import gc, torch
import matplotlib.pyplot as plt
from solution import validate_model
%matplotlib inline

device = "0" if torch.cuda.is_available() else "cpu"
print(f"[INFO] ใช้อุปกรณ์: {device}")

print("\n--- เริ่มการตรวจสอบ ---")
metrics = validate_model("yolo11n.pt", "coco8.yaml")

print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall:    {metrics.box.mr:.4f}")
print(f"  mAP@0.5:   {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")

print("\n  ผลแต่ละคลาส:")
names = metrics.names
for idx, name in names.items():
    p, r, ap50, ap = metrics.box.class_result(idx)
    print(f"    [{idx:2d}] {name:<20} P={p:.3f} R={r:.3f} AP50={ap50:.3f} AP={ap:.3f}")
print("--- สิ้นสุดการตรวจสอบ ---")

cls_names = list(names.values())
ap50_vals = [metrics.box.class_result(i)[2] for i in names.keys()]
plt.figure(figsize=(10,4))
plt.bar(cls_names, ap50_vals, color="#2ecc71", edgecolor="black", linewidth=0.5)
plt.axhline(metrics.box.map50, color="red", linestyle="--", linewidth=1.5,
            label=f"Mean mAP@0.5 = {metrics.box.map50:.3f}")
plt.xlabel("คลาส"); plt.ylabel("AP@0.5")
plt.title("AP@0.5 แต่ละคลาส (yolo11n / coco8)")
plt.xticks(rotation=45, ha="right"); plt.legend(); plt.tight_layout(); plt.show()

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
